In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ISLP import load_data

from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score,
    log_loss
)

smarket = load_data("Smarket")
smarket

,Year,Lag1,Lag2,Lag3,Lag4,Lag5,Volume,Today,Direction
0,2001,0.381,-0.192,-2.624,-1.055,5.010,1.19130,0.959,Up
1,2001,0.959,0.381,-0.192,-2.624,-1.055,1.29650,1.032,Up
2,2001,1.032,0.959,0.381,-0.192,-2.624,1.41120,-0.623,Down
3,2001,-0.623,1.032,0.959,0.381,-0.192,1.27600,0.614,Up
4,2001,0.614,-0.623,1.032,0.959,0.381,1.20570,0.213,Up
...,...,...,...,...,...,...,...,...,...
1245,2005,0.422,0.252,-0.024,-0.584,-0.285,1.88850,0.043,Up
1246,2005,0.043,0.422,0.252,-0.024,-0.584,1.28581,-0.955,Down
1247,2005,-0.955,0.043,0.422,0.252,-0.024,1.54047,0.130,Up
1248,2005,0.130,-0.955,0.043,0.422,0.252,1.42236,-0.298,Down


In [2]:
train = smarket["Year"] < 2005
test = smarket["Year"] == 2005

features = ["Lag1", "Lag2"]
X_train = smarket.loc[train, features]
X_test = smarket.loc[test, features]

y_train = smarket.loc[train, "Direction"]
y_test = smarket.loc[test, "Direction"]

print(smarket["Direction"].value_counts())
print(smarket["Direction"].value_counts(normalize=True))

# majority-class baseline

Direction
Up      648
Down    602
Name: count, dtype: int64
Direction
Up      0.5184
Down    0.4816
Name: proportion, dtype: float64


In [3]:
logit = LogisticRegression()
logit.fit(X_train, y_train)

print(logit.classes_)   # log(P(up) / P(down)) = coef * X + intercept
print(logit.coef_)
print(logit.intercept_)

# coefs are not significant

['Down' 'Up']
[[-0.05552614 -0.0442976 ]]
[0.03222297]


In [4]:
logit_pred = logit.predict(X_test)
logit_prob = logit.predict_proba(X_test)

result = pd.DataFrame({
    "actual": y_test,
    "predicted": logit_pred,
    "prob_down": logit_prob[:, 0],
    "prob_up": logit_prob[:, 1]
})

print((y_test == logit_pred).value_counts(normalize=True))
result

Direction
True     0.559524
False    0.440476
Name: proportion, dtype: float64


,actual,predicted,prob_down,prob_up
998,Down,Up,0.490174,0.509826
999,Down,Up,0.479200,0.520800
1000,Down,Up,0.466801,0.533199
1001,Up,Up,0.474005,0.525995
1002,Down,Up,0.492797,0.507203
...,...,...,...,...
1245,Up,Down,0.500593,0.499407
1246,Down,Up,0.497215,0.502785
1247,Up,Up,0.479176,0.520824
1248,Down,Up,0.483179,0.516821


In [5]:
# linear discriminant analysis

lda = LinearDiscriminantAnalysis(store_covariance=True)
lda.fit(X_train, y_train)

print(lda.classes_)   # log(P(up) / P(down)) = coef * X + intercept
print(lda.coef_)
print(lda.intercept_)

# results very similar to logistic regression

['Down' 'Up']
[[-0.05544078 -0.0443452 ]]
[0.03221375]


In [6]:
print(lda.classes_)
print(lda.priors_)  # frequency of classes
print(lda.means_)   # mean(Lag1, Lag2) given classes
print(lda.covariance_)  # cov matrix is independent of classes

['Down' 'Up']
[0.49198397 0.50801603]
[[ 0.04279022  0.03389409]
 [-0.03954635 -0.03132544]]
[[ 1.50886781 -0.03340234]
 [-0.03340234  1.5095363 ]]


In [7]:
print("Logistic Regression")
print(confusion_matrix(y_test, logit_pred))

lda_pred = lda.predict(X_test)

print("\nLDA")
print(confusion_matrix(y_test, lda_pred))

# same prediction results

Logistic Regression
[[ 35  76]
 [ 35 106]]

LDA
[[ 35  76]
 [ 35 106]]
